In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="18Dw5o8G8OWnZXYUom7PR64pUpNlnHEjI", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/02_01_intro.mp3"))

# 🚀 Custom Slash Commands & Skills — Building Reusable Workflows

In this notebook, we will build a working simulator for Claude Code's custom command and skill system. You will create project commands, personal commands, and skills with full SKILL.md frontmatter parsing — including `context:fork`, `allowed-tools`, and `argument-hint`.

By the end, you will be able to:
- Create and parse slash commands from `.claude/commands/`
- Build skills with SKILL.md metadata (context:fork, allowed-tools, argument-hint)
- Understand when to use commands vs skills vs CLAUDE.md rules
- Simulate context:fork isolation in a sub-agent

# 🤖 AI Teaching Assistant

Need help with this notebook? Open the **AI Teaching Assistant** — it has already read this entire notebook and can help with concepts, code, and exercises.

**[👉 Open AI Teaching Assistant](https://pods.vizuara.ai/courses/claude-certified-architect/claude-code-workflows/practice/2/assistant)**

*Tip: Open it in a separate tab and work through this notebook side-by-side.*


In [ ]:
#@title 🎧 Listen: Why Matters
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_02_why_matters.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 1. Why Does This Matter?

CLAUDE.md tells Claude *how* to behave — it is background configuration. But what about **repeatable workflows** that you trigger on demand?

"Generate a new API endpoint with tests, validation, and OpenAPI docs."
"Run the full deployment checklist."
"Create a database migration with up/down scripts."

These are not background rules — they are foreground actions. That is what commands and skills are for. The exam tests whether you can choose the right abstraction and configure it properly.

In [ ]:
#@title 🎧 Listen: Setup
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_03_setup.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
import os
import json
import tempfile
import shutil
import re
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any

WORKSPACE = tempfile.mkdtemp(prefix="claude_commands_lab_")
print(f"Workspace: {WORKSPACE}")

In [ ]:
#@title 🎧 Listen: Intuition
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_04_intuition.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 2. Building Intuition — Commands vs Skills

Think of it like cooking:
- **CLAUDE.md rules** = Your kitchen's house rules. "Always wash hands. Always use metric measurements." These are always active.
- **Slash commands** = Recipes on index cards. "How to make pasta: boil water, add pasta, drain after 8 minutes..." Simple, step-by-step instructions you follow when needed.
- **Skills** = A sous chef you can delegate to. They take the recipe, go into their own workspace (`context:fork`), use only the tools you allow (`allowed-tools`), and return the finished dish. More powerful, more isolated.

Let us build each one.

In [ ]:
#@title 🎧 Listen: Commands Walkthrough
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_05_commands_walkthrough.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
@dataclass
class SlashCommand:
    """A parsed slash command from .claude/commands/."""
    name: str              # Derived from filename (e.g., "new-api-endpoint")
    content: str           # The prompt template
    scope: str             # "project" or "personal"
    source_path: str       # Where the file lives
    has_arguments: bool    # Whether it uses $ARGUMENTS placeholder

    def render(self, arguments: str = "") -> str:
        """Render the command with the given arguments."""
        return self.content.replace("$ARGUMENTS", arguments)


def parse_command_file(path: str, scope: str) -> SlashCommand:
    """Parse a .md file from .claude/commands/ into a SlashCommand."""
    name = os.path.splitext(os.path.basename(path))[0]
    with open(path) as f:
        content = f.read()

    return SlashCommand(
        name=name,
        content=content,
        scope=scope,
        source_path=path,
        has_arguments="$ARGUMENTS" in content,
    )


# Create project commands
project_commands_dir = os.path.join(WORKSPACE, ".claude", "commands")
os.makedirs(project_commands_dir, exist_ok=True)

commands = {
    "new-api-endpoint.md": """Create a new API endpoint with the following specifications:

1. Create the route handler in `src/api/v2/$ARGUMENTS.ts`
2. Add input validation using Zod schemas
3. Wrap the handler in `withErrorHandler` middleware
4. Create a test file at `src/api/v2/$ARGUMENTS.test.ts`
5. Add the route to the OpenAPI spec in `docs/openapi.yaml`
6. Register the route in `src/api/v2/index.ts`

Follow all conventions in CLAUDE.md and the api-conventions rules.
""",

    "run-review.md": """Review the current changes for:

1. Type safety — are all types explicit? Any `any` usage?
2. Error handling — are all error paths covered?
3. Test coverage — do new functions have tests?
4. Naming — do names follow project conventions?
5. Security — any injection risks, exposed secrets, or missing validation?

Output a structured review with severity levels: critical, warning, info.
""",

    "generate-migration.md": """Generate a database migration named `$ARGUMENTS`:

1. Create file: `migrations/{timestamp}_$ARGUMENTS.sql`
2. Include both UP and DOWN sections
3. Follow column naming conventions (snake_case)
4. Add indexes for foreign keys
5. Create a test in `migrations/tests/$ARGUMENTS.test.sql`
""",
}

for fname, content in commands.items():
    with open(os.path.join(project_commands_dir, fname), "w") as f:
        f.write(content)

# Create personal commands
personal_commands_dir = os.path.join(WORKSPACE, "_user_home", ".claude", "commands")
os.makedirs(personal_commands_dir, exist_ok=True)

personal_commands = {
    "explain-simple.md": """Explain the following code or concept in simple terms.
Use analogies and avoid jargon. Assume I am a junior developer.

Topic: $ARGUMENTS
""",
}

for fname, content in personal_commands.items():
    with open(os.path.join(personal_commands_dir, fname), "w") as f:
        f.write(content)

print("Created commands:")
print(f"  Project ({len(commands)}):")
for name in commands:
    print(f"    /{ os.path.splitext(name)[0] }")
print(f"  Personal ({len(personal_commands)}):")
for name in personal_commands:
    print(f"    /{ os.path.splitext(name)[0] }")

In [ ]:
#@title 🎧 Listen: Load Commands Walkthrough
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_06_load_commands_walkthrough.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Load and test commands
def load_commands(project_root: str, user_home: str) -> Dict[str, SlashCommand]:
    """Load all available slash commands from both project and personal locations."""
    commands = {}

    # Project commands (shared via git)
    proj_dir = os.path.join(project_root, ".claude", "commands")
    if os.path.isdir(proj_dir):
        for fname in sorted(os.listdir(proj_dir)):
            if fname.endswith(".md"):
                path = os.path.join(proj_dir, fname)
                cmd = parse_command_file(path, "project")
                commands[cmd.name] = cmd

    # Personal commands (NOT shared)
    personal_dir = os.path.join(user_home, ".claude", "commands")
    if os.path.isdir(personal_dir):
        for fname in sorted(os.listdir(personal_dir)):
            if fname.endswith(".md"):
                path = os.path.join(personal_dir, fname)
                cmd = parse_command_file(path, "personal")
                # Personal overrides project if same name
                commands[cmd.name] = cmd

    return commands


all_commands = load_commands(WORKSPACE, os.path.join(WORKSPACE, "_user_home"))

print("Available commands:")
print("-" * 60)
for name, cmd in sorted(all_commands.items()):
    args_note = "(takes arguments)" if cmd.has_arguments else "(no arguments)"
    print(f"  /{name:25s} [{cmd.scope:8s}] {args_note}")

# Render a command with arguments
print("\n" + "=" * 60)
print("Rendered: /new-api-endpoint users/preferences")
print("=" * 60)
rendered = all_commands["new-api-endpoint"].render("users/preferences")
print(rendered)

In [ ]:
#@title 🎧 Listen: Skills Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_07_skills_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 3. Skills — Commands with Superpowers

Skills add three capabilities that plain commands lack:
1. **context:fork** — Run in an isolated sub-agent context
2. **allowed-tools** — Restrict which tools the skill can use
3. **argument-hint** — Self-documenting argument description

Let us build the skill parser.

In [ ]:
#@title 🎧 Listen: Skills Walkthrough
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_08_skills_walkthrough.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
@dataclass
class Skill:
    """A parsed skill with SKILL.md frontmatter metadata."""
    name: str
    content: str
    scope: str              # "project" or "personal"
    source_path: str
    context_fork: bool      # If True, runs in isolated sub-agent
    allowed_tools: List[str]  # Which tools the skill can use
    argument_hint: str      # Description of expected arguments
    has_arguments: bool

    def render(self, arguments: str = "") -> str:
        return self.content.replace("$ARGUMENTS", arguments)


def parse_skill(skill_dir: str, scope: str) -> Optional[Skill]:
    """Parse a skill directory containing SKILL.md."""
    skill_md_path = os.path.join(skill_dir, "SKILL.md")
    if not os.path.exists(skill_md_path):
        return None

    with open(skill_md_path) as f:
        raw = f.read()

    # Parse YAML frontmatter
    context_fork = False
    allowed_tools = []
    argument_hint = ""

    lines = raw.split("\n")
    body_start = 0

    if lines[0].strip() == "---":
        for i in range(1, len(lines)):
            if lines[i].strip() == "---":
                body_start = i + 1
                break

        frontmatter = lines[1:body_start - 1]
        for line in frontmatter:
            line = line.strip()
            if line.startswith("context:"):
                context_fork = "fork" in line
            elif line.startswith("argument-hint:"):
                argument_hint = line.split(":", 1)[1].strip().strip('"')
            elif line.startswith("- "):
                # Part of allowed-tools list
                allowed_tools.append(line[2:].strip())
            elif line.startswith("allowed-tools:"):
                # Might be inline list or start of list
                pass

    body = "\n".join(lines[body_start:])
    name = os.path.basename(skill_dir)

    return Skill(
        name=name,
        content=body,
        scope=scope,
        source_path=skill_dir,
        context_fork=context_fork,
        allowed_tools=allowed_tools,
        argument_hint=argument_hint,
        has_arguments="$ARGUMENTS" in body,
    )


# Create example skills
skills_dir = os.path.join(WORKSPACE, ".claude", "skills")

skill_definitions = {
    "generate-migration": {
        "SKILL.md": """---
context: fork
allowed-tools:
  - read
  - write
  - bash
argument-hint: "migration name (e.g., add-user-preferences-table)"
---

# Generate Database Migration

Create a new database migration with the given name: $ARGUMENTS

1. Create migration file at `migrations/{timestamp}_$ARGUMENTS.sql`
2. Include UP and DOWN sections
3. Follow column naming conventions from .claude/rules/database.md
4. Add a corresponding test in `migrations/tests/`
5. Update the migration index
"""
    },
    "deploy-staging": {
        "SKILL.md": """---
context: fork
allowed-tools:
  - bash
  - read
argument-hint: "version tag (e.g., v1.2.3)"
---

# Deploy to Staging

Deploy version $ARGUMENTS to the staging environment:

1. Run the full test suite
2. Build the production bundle
3. Run database migrations
4. Deploy to staging cluster
5. Run smoke tests
6. Report deployment status
"""
    },
    "review-checklist": {
        "SKILL.md": """---
context: fork
allowed-tools:
  - read
  - grep
argument-hint: "PR number or branch name"
---

# Code Review Checklist

Review $ARGUMENTS against the team's quality standards:

1. Check for type safety issues (any `any` types?)
2. Verify error handling completeness
3. Check test coverage for new code
4. Validate naming conventions
5. Scan for security vulnerabilities
6. Verify API contract compatibility
"""
    },
}

for skill_name, files in skill_definitions.items():
    skill_path = os.path.join(skills_dir, skill_name)
    os.makedirs(skill_path, exist_ok=True)
    for fname, content in files.items():
        with open(os.path.join(skill_path, fname), "w") as f:
            f.write(content)

# Parse all skills
parsed_skills = {}
for skill_name in os.listdir(skills_dir):
    skill_path = os.path.join(skills_dir, skill_name)
    if os.path.isdir(skill_path):
        skill = parse_skill(skill_path, "project")
        if skill:
            parsed_skills[skill.name] = skill

print("Parsed Skills:")
print("=" * 65)
for name, skill in sorted(parsed_skills.items()):
    print(f"\n🔧 /{name}")
    print(f"   context:fork = {skill.context_fork}")
    print(f"   allowed-tools = {skill.allowed_tools}")
    print(f"   argument-hint = {skill.argument_hint!r}")
    print(f"   scope = {skill.scope}")

In [ ]:
#@title 🎧 Listen: Fork Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_09_fork_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 4. Simulating context:fork — Sub-Agent Isolation

The most important skill property is `context:fork`. Let us simulate what happens with and without it.

In [ ]:
#@title 🎧 Listen: Fork Walkthrough
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_10_fork_walkthrough.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
@dataclass
class ConversationContext:
    """Simulates Claude Code's conversation context window."""
    messages: List[str] = field(default_factory=list)
    token_count: int = 0

    def add_message(self, content: str, role: str = "assistant"):
        self.messages.append(f"[{role}]: {content}")
        self.token_count += len(content.split())  # Rough approximation

    def summary(self) -> str:
        return f"Context: {len(self.messages)} messages, ~{self.token_count} tokens"


def execute_without_fork(skill: Skill, arguments: str, context: ConversationContext):
    """Execute skill in the MAIN conversation context (no fork)."""
    prompt = skill.render(arguments)
    context.add_message(f"Executing /{skill.name} {arguments}...", "user")

    # Simulated verbose output from skill execution
    steps = [
        f"Reading project structure... found 847 files",
        f"Scanning migrations directory... found 23 existing migrations",
        f"Generating migration: {arguments}",
        f"Creating UP section with 3 tables, 7 columns, 2 indexes",
        f"Creating DOWN section with DROP TABLE statements",
        f"Writing test file with 5 test cases",
        f"Updating migration index... done",
    ]

    for step in steps:
        context.add_message(step)

    context.add_message(f"Migration '{arguments}' created successfully!")
    return context


def execute_with_fork(skill: Skill, arguments: str, main_context: ConversationContext):
    """Execute skill in a FORKED sub-agent context."""
    # Create isolated context for the sub-agent
    sub_context = ConversationContext()
    prompt = skill.render(arguments)

    # All the verbose work happens in the sub-agent's context
    steps = [
        f"Reading project structure... found 847 files",
        f"Scanning migrations directory... found 23 existing migrations",
        f"Generating migration: {arguments}",
        f"Creating UP section with 3 tables, 7 columns, 2 indexes",
        f"Creating DOWN section with DROP TABLE statements",
        f"Writing test file with 5 test cases",
        f"Updating migration index... done",
    ]

    for step in steps:
        sub_context.add_message(step)

    # Only the RESULT goes back to the main context
    result = f"✅ Migration '{arguments}' created: migrations/20250315_add_user_prefs.sql"
    main_context.add_message(result)

    return main_context, sub_context


# Compare the two approaches
print("WITHOUT context:fork")
print("=" * 50)
ctx_no_fork = ConversationContext()
ctx_no_fork.add_message("Previous conversation message 1")
ctx_no_fork.add_message("Previous conversation message 2")
print(f"Before: {ctx_no_fork.summary()}")

skill = parsed_skills["generate-migration"]
ctx_no_fork = execute_without_fork(skill, "add-user-prefs", ctx_no_fork)
print(f"After:  {ctx_no_fork.summary()}")
print(f"⚠️ All {len(ctx_no_fork.messages)} messages are in your main context!\n")

print("WITH context:fork")
print("=" * 50)
ctx_with_fork = ConversationContext()
ctx_with_fork.add_message("Previous conversation message 1")
ctx_with_fork.add_message("Previous conversation message 2")
print(f"Before: {ctx_with_fork.summary()}")

ctx_with_fork, sub_ctx = execute_with_fork(skill, "add-user-prefs", ctx_with_fork)
print(f"After:  {ctx_with_fork.summary()}")
print(f"Sub-agent: {sub_ctx.summary()}")
print(f"✅ Main context stays clean — only the result was added!")

In [ ]:
#@title 🎧 Listen: Fork Thinking
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_11_fork_thinking.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

This is the key insight: `context:fork` prevents verbose skill execution from consuming your main conversation's context window. The sub-agent does all the work in isolation and returns just the result.

---

### 🤔 Think About This

When would you NOT want context:fork?

Answer: When you need the skill's intermediate output to inform your next interactive steps. For example, a "debug this error" command where you want to see the investigation details to guide further work. Fork is best for fire-and-forget workflows.

In [ ]:
#@title 🎧 Listen: Todo Decision
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_12_todo_decision.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 5. 🔧 Your Turn — Decision Matrix

Complete this decision matrix for when to use each abstraction.

In [ ]:
# ============ TODO ============
# For each scenario, choose the best abstraction:
# "claude_md" = CLAUDE.md rule (passive, always active)
# "command"   = Slash command (simple template, triggered on demand)
# "skill"     = Skill with SKILL.md (isolated execution, tool restrictions)

scenarios = {
    "Team uses 2-space indentation everywhere": "???",
    "Generate a new React component with tests": "???",
    "Deploy to production with safety checks": "???",
    "Always validate API inputs with Zod": "???",
    "Run a 12-step security audit checklist": "???",
    "Explain code in simple terms (personal)": "???",
}
# ==============================

In [ ]:
# ✅ Verification
correct_answers = {
    "Team uses 2-space indentation everywhere": "claude_md",
    "Generate a new React component with tests": "command",
    "Deploy to production with safety checks": "skill",
    "Always validate API inputs with Zod": "claude_md",
    "Run a 12-step security audit checklist": "skill",
    "Explain code in simple terms (personal)": "command",
}

explanations = {
    "Team uses 2-space indentation everywhere": "Passive rule, always active → CLAUDE.md",
    "Generate a new React component with tests": "Simple template workflow → Command",
    "Deploy to production with safety checks": "Needs isolation + restricted tools → Skill with context:fork",
    "Always validate API inputs with Zod": "Passive rule, always active → CLAUDE.md",
    "Run a 12-step security audit checklist": "Complex workflow, should be isolated → Skill with context:fork",
    "Explain code in simple terms (personal)": "Simple personal template → Personal command in ~/.claude/commands/",
}

correct = 0
for scenario, answer in scenarios.items():
    expected = correct_answers[scenario]
    if answer.strip().lower() == expected:
        print(f"✅ '{scenario}' → {expected}. {explanations[scenario]}")
        correct += 1
    else:
        print(f"❌ '{scenario}' → Expected '{expected}', got '{answer}'. {explanations[scenario]}")

print(f"\nScore: {correct}/{len(scenarios)}")

In [ ]:
#@title 🎧 Listen: Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_13_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 6. 📊 Visualization — Command & Skill Feature Comparison

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(10, 5))

features = [
    "Shared via Git",
    "Takes Arguments",
    "Context Isolation",
    "Tool Restrictions",
    "Self-Documenting",
    "Multi-Step Workflow",
]

# Score each feature for each abstraction (0=no, 0.5=partial, 1=yes)
claude_md = [1, 0, 0, 0, 0.5, 0]
commands  = [1, 1, 0, 0, 0.5, 1]
skills    = [1, 1, 1, 1, 1, 1]

x = np.arange(len(features))
width = 0.25

bars1 = ax.bar(x - width, claude_md, width, label='CLAUDE.md Rules', color='#3498db')
bars2 = ax.bar(x, commands, width, label='Slash Commands', color='#2ecc71')
bars3 = ax.bar(x + width, skills, width, label='Skills', color='#e67e22')

ax.set_ylabel('Support Level')
ax.set_title('Feature Comparison: CLAUDE.md vs Commands vs Skills', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(features, rotation=20, ha='right', fontsize=9)
ax.set_yticks([0, 0.5, 1])
ax.set_yticklabels(['No', 'Partial', 'Yes'])
ax.legend()
ax.set_ylim(0, 1.3)

plt.tight_layout()
plt.savefig("commands_vs_skills.png", dpi=150, bbox_inches='tight')
plt.show()
print("📊 Feature comparison chart generated!")

In [ ]:
#@title 🎧 Listen: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_14_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 7. Reflection and Next Steps

**Key takeaways:**
1. **CLAUDE.md rules** = passive, always-on conventions. "Use Vitest." "Validate with Zod."
2. **Slash commands** = triggered prompt templates. Project-scoped (shared) or personal.
3. **Skills** = commands with metadata: `context:fork` for isolation, `allowed-tools` for restrictions, `argument-hint` for documentation.
4. `context:fork` prevents verbose skill execution from consuming your main conversation context.
5. If your CLAUDE.md instruction starts with "When the developer asks you to..." — extract it into a skill.

**Exam tip:** The exam loves scenario questions: "Which abstraction should you use for X?" Use the decision matrix from this notebook.

In [ ]:
# Cleanup
shutil.rmtree(WORKSPACE)
print("✅ Workspace cleaned up. Notebook complete!")